# 1. От баланса фотонов к измеряемому сигналу

Начинаем с одного мгновенно испущенного фотона и однородной среды.
После этой работы должны быть понятны размерности уравнения переноса,
нормировка фазовой функции и различие плотности фотонов, скалярного потока
и ожидаемого числа регистраций. Все числа ниже — учебные параметры.

In [ ]:
from pathlib import Path
import sys, time, platform
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Notebook can be started from the repo root or notebooks/course.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/lighthit').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start this notebook inside the LightHit repository')
sys.path.insert(0, str(ROOT / 'src'))
from lighthit import Medium

# Explicit public test medium. No private provider is imported.
medium = Medium(0.04, 0.05, 0.7, 1.35, 450.0, 'course-synthetic')
np.set_printoptions(precision=7, suppress=True)
print('Python:', sys.executable)
print('Platform:', platform.platform())

## 1.1. Что находится в элементе фазового пространства

Пусть $f(\mathbf x,\mathbf s,t)\,d^3x\,d\Omega$ — ожидаемое число фотонов,
где $\mathbf s$ — единичное направление. Скорость $v=c_0/n_g$, интенсивность
$I=vf$. За $dt$ фотон смещается на $v\mathbf s\,dt$; вероятность поглощения
равна $v\mu_a\,dt$, рассеяния — $v\mu_s\,dt$ с точностью до $O(dt^2)$.

Вычитая уход и добавляя приход из всех направлений, получаем
$$
\frac1v\partial_t I+\mathbf s\cdot\nabla I+
(\mu_a+\mu_s)I-\mu_s\int p(\mathbf s\cdot\mathbf s')I(\mathbf s')d\Omega'=q.
$$
$\mu_a,\mu_s$ имеют размерность м$^{-1}$; $p$ нормирована интегралом по сфере
к единице. $q\,d^3x\,dt\,d\Omega$ — число испущенных фотонов.

Для HG-функции
$$p(x)=\frac{1-g^2}{4\pi(1+g^2-2gx)^{3/2}},\quad -1<g<1.$$
Проверим нормировку и средний косинус: $2\pi\int p(x)dx=1$,
$2\pi\int xp(x)dx=g$.

## 1.2. Индикатрисса Henyey--Greenstein

Уравнение переноса верно для любой нормированной индикатриссы. Модель,
используемая всюду в книге, — Henyey--Greenstein с явно заданным $g$:
$$p_g(c)=\frac{1-g^2}{4\pi(1+g^2-2gc)^{3/2}},\qquad c=\mathbf s\cdot\mathbf s'.$$
Проверяем две величины, на которых держатся все последующие сокращения:
нормировку $\int p_g\,d\Omega=1$ и средний косинус $\int c\,p_g\,d\Omega=g$.
Остаток квадратуры зависит от окружения и растёт с $g$ — это свойство правила,
а не формулы.


In [ ]:
from numpy.polynomial.legendre import leggauss
from scipy.special import eval_legendre
from lighthit.single import hg_phase

# Квадратура в cos(theta): при g=0.9 ядро острое, узлов нужно много.
x, w = leggauss(512)
fig, ax = plt.subplots(figsize=(6, 4))
rows = []
for g in (0.0, 0.3, 0.6, 0.9):
    p = hg_phase(x, g)
    norm = 2 * np.pi * float(np.dot(w, p))
    mean = 2 * np.pi * float(np.dot(w, x * p))
    rows.append((g, norm - 1.0, mean - g))
    ax.semilogy(x, p, label=f'g = {g}')
    assert abs(norm - 1) < 1e-10 and abs(mean - g) < 1e-10
ax.set(xlabel='cos(угол рассеяния)', ylabel='p [ср^-1]',
       title='Henyey-Greenstein при разных g')
ax.legend(); plt.show()

display(Markdown('| g | ошибка нормировки | ошибка среднего косинуса |\n|--:|--:|--:|\n'
                 + '\n'.join(f'| {g} | {dn:.2e} | {dm:.2e} |' for g, dn, dm in rows)))


### Собственные значения по Лежандру

Оператор рассеяния диагонален по угловому индексу, и его собственные значения
равны $g^\ell$:
$$2\pi\int_{-1}^{1}p_g(c)P_\ell(c)\,dc=g^\ell.$$
Именно эта форма, а не сама $p_g$, используется решателем. Переход к
сферическим гармоникам — глава 5.


In [ ]:
g = 0.9
p = hg_phase(x, g)
degrees = np.arange(9)
measured = np.array([2 * np.pi * float(np.dot(w, eval_legendre(int(l), x) * p))
                     for l in degrees])
expected = g ** degrees
print('l  measured        g^l             |diff|')
for l, a, b in zip(degrees, measured, expected):
    print(f'{l}  {a:.12f}  {b:.12f}  {abs(a-b):.2e}')
assert np.max(np.abs(measured - expected)) < 1e-9


## 1.3. Закон сохранения

Интеграл члена рассеяния по выходному направлению равен
$\mu_s\int I\,d\Omega$: он сокращает уход из направления.
После интегрирования также по всему пространству поток на бесконечности
исчезает. Для одиночной вспышки:
$$N(t)=\int f\,d^3x\,d\Omega=e^{-\mu_a vt}.$$
Сохранение $N$ при $\mu_a=0$ не означает, что плотность в конкретной точке
не меняется. Рассеяние переносит её в другие точки и направления.

In [ ]:
t=np.linspace(0,500,301)
N=np.exp(-medium.absorption_per_m*medium.speed_m_per_ns*t)
fig,ax=plt.subplots();ax.plot(t,N);ax.set(xlabel='t [ns]',ylabel='surviving photon number');plt.show()

## 1.4. Два показателя преломления

Скорость переноса задаёт групповой показатель $n_g$, а угол и выход черенковского
источника — фазовый $n_{\rm ph}$. Монохроматическая среда `Medium`, которую
получает решатель, хранит `group_index` и **не** хранит фазовый показатель.
Ниже — цена подстановки одного вместо другого.


In [ ]:
from lighthit import SpectralMedium
from lighthit.medium import C_VACUUM_M_PER_NS

# Публичная синтетическая таблица главы 2, не калибровка.
spectral = SpectralMedium([400., 450., 500., 550.],
                          [.030, .021, .038, .090],
                          [.030, .022, .017, .013],
                          [1.3435, 1.3390, 1.3360, 1.3340],
                          [1.3860, 1.3740, 1.3670, 1.3630],
                          g=.9, provenance='synthetic, notebook')
probe = 462.5
sampled = spectral.sample(probe)
band = spectral.band(probe)
print('поля монохроматической среды:', sorted(band.__dataclass_fields__))
assert 'phase_index' not in band.__dataclass_fields__

distance = np.linspace(0., 300., 301)
dt = distance * (float(sampled['group_index']) - float(sampled['phase_index'])) / C_VACUUM_M_PER_NS
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(distance, dt)
ax.axhline(20., ls='--', lw=1, label='один временной бин 20 нс')
ax.set(xlabel='расстояние [м]', ylabel='ошибка времени при замене n_g на n_ph [нс]')
ax.legend(); plt.show()
print(f'на 100 м ошибка {100*(float(sampled["group_index"])-float(sampled["phase_index"]))/C_VACUUM_M_PER_NS:.3f} нс')


## Задания

1. Восстановить вывод RTE из баланса в прямоугольном объёме; указать порядок
отброшенных по $dt$ величин. Проверить размерность каждого члена.
2. Вывести уравнение для плотности $f_\lambda=I_\lambda/v_g$ и объяснить, почему
множитель $1/v_g$ стоит перед производной по времени в одной форме и
отсутствует в другой.
3. Показать аналитически, что уход и приход рассеяния сокращаются при
интегрировании по выходному направлению для любой нормированной индикатриссы.
4. Взять предел $g\to0$ и убедиться, что остаётся единственное ненулевое
собственное значение. Построить поле, в котором $p=1/4\pi$, но $I$ всё ещё
зависит от направления.
5. Найти расстояние, на котором ошибка от подстановки $n_{\rm ph}$ вместо $n_g$
достигает одного временного бина 20 нс.
6. Повторить измерение остатка квадратуры при $g=0.95$ и $g=0.99$ и оценить,
сколько узлов нужно, чтобы удержать его на прежнем уровне.

Код: `medium.py`, `model.py`, `single.py`; текст книги: глава 2.
